# Genome-resolved metagenomic dataset of 110 healthy Indian gut microbiomes with probiotic genome annotations and vitamin biosynthesis profiles Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.zhh6-c0tb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access `metadata` as a data class, print name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"\nVersion: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.date_published}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available RecordSets by @id and their fields
print("Available RecordSets and their fields:")
record_set_objs = list(dataset.record_sets)
if not record_set_objs:
    print("No RecordSets found in this dataset.")
else:
    for rs in record_set_objs:
        print(f"\nRecordSet @id: {rs.id}")
        print(f"  Name: {rs.name if hasattr(rs, 'name') else ''}")
        print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id} | Name: {field.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# ---
# Find all record set @ids for use in extraction (if available)
records_dataframes = {}
record_set_ids = []

record_set_objs = list(dataset.record_sets)
if not record_set_objs:
    print("No RecordSet entries defined in this dataset. Please check the dataset schema.")
else:
    record_set_ids = [rs.id for rs in record_set_objs]
    pprint(f"All RecordSet @id values: {record_set_ids}")
    # Extract data from each record set
    for record_set_id in record_set_ids:
        print(f"\nLoading records from RecordSet {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if not records:
            print(f"  No records found for {record_set_id}.")
            continue
        df = pd.DataFrame(records)
        records_dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head())
    if len(records_dataframes) == 0:
        print("No records loaded for any RecordSet.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA operations
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Pick the first loaded DataFrame for demonstration (or specify if known)
if not records_dataframes:
    print("No tabular DataFrame loaded; EDA cannot be performed.")
else:
    record_set_id = list(records_dataframes.keys())[0]
    df = records_dataframes[record_set_id]
    print(f"Running EDA on RecordSet: {record_set_id}\nColumns: {df.columns.tolist()}")
    
    # Attempt to pick a numeric field (float or int)
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Try to coerce columns to numeric to find candidates
        non_numeric = []
        for col in df.columns:
            try:
                df[col + '_num'] = pd.to_numeric(df[col], errors='coerce')
                if df[col + '_num'].notnull().any():
                    numeric_candidates.append(col)
            except:
                non_numeric.append(col)
        if not numeric_candidates:
            print("No numeric fields detected for EDA.")
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        field_to_use = numeric_field if numeric_field in df.columns else numeric_field + '_num'
        print(f"Using numeric field: {field_to_use}")
        threshold = df[field_to_use].mean() if df[field_to_use].notnull().any() else 0
        filtered_df = df[df[field_to_use] > threshold]
        print(f"\nFiltered records with {field_to_use} > mean:")
        print(filtered_df.head())
        filtered_df[field_to_use + "_normalized"] = (filtered_df[field_to_use] - filtered_df[field_to_use].mean()) / filtered_df[field_to_use].std()
        print(f"\nNormalized {field_to_use} for filtered records:")
        print(filtered_df[[field_to_use, field_to_use + "_normalized"]].head())
        # Try to pick a categorical field for grouping
        non_numeric_fields = [c for c in df.columns if c not in numeric_candidates]
        group_field = non_numeric_fields[0] if non_numeric_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (mean of numeric fields):")
            print(grouped_df.head())
    else:
        print("No suitable numeric fields to process.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic visualization: Histogram or bar plot
import matplotlib.pyplot as plt
import seaborn as sns

if not records_dataframes:
    print("No tables loaded for visualization.")
else:
    # Pick first DataFrame and a numeric field if available
    df = list(records_dataframes.values())[0]
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        field = numeric_candidates[0]
        plt.figure(figsize=(6, 3))
        sns.histplot(df[field].dropna(), kde=True)
        plt.title(f'Distribution of {field}')
        plt.xlabel(field)
        plt.show()
    else:
        # Plot bar plot of a categorical field
        cat_fields = [c for c in df.columns if df[c].nunique() < 20]
        if cat_fields:
            field = cat_fields[0]
            df[field].value_counts().plot(kind='bar', figsize=(6,3))
            plt.title(f'Value counts of {field}')
            plt.ylabel('Count')
            plt.show()
        else:
            print("No numeric or low-cardinality categorical fields available to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

The notebook demonstrated how to load and explore a FAIR²-compliant metagenomic dataset using the `mlcroissant` library. Carefully inspect the record sets, fields (@id), and data types to adapt EDA and visualizations to the actual data structure in your use case. For more complex analysis, consult the Croissant schema for advanced table relations and metadata details.